# Getting a Kernel ready...

In [ ]:
#r "nuget: Microsoft.DotNet.Interactive.SqlServer, *-*"

In [ ]:
#!connect mssql --kernel-name sql2025 --connection-string "Server=sql2025;TrustServerCertificate=True;Integrated Security=True"

# Check our connection

In [ ]:
SELECT @@VERSION

# Create a sample DB and get it ready

In [ ]:
USE master
IF EXISTS (SELECT name FROM sys.databases WHERE name='StorageIsExpensive')BEGIN
    ALTER DATABASE StorageIsExpensive SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
    DROP DATABASE StorageIsExpensive;
END
GO
CREATE DATABASE StorageIsExpensive
GO
USE StorageIsExpensive
GO

In [ ]:
create master key encryption by password = 'MyTest!Mast3rP4ss'


In [ ]:
sp_configure 'external rest endpoint enabled', 1;
RECONFIGURE WITH OVERRIDE

In [ ]:
IF OBJECT_ID('dbo.ConferenceSessions', 'U') IS NOT NULL DROP TABLE dbo.ConferenceSessions;
CREATE TABLE [dbo].[ConferenceSessions] ([id] int NOT NULL,
[Title] [nvarchar](4000) NULL,
[description] [nvarchar](4000) NULL,
MainSpeaker [nvarchar](4000) NULL,
[Speakers] [nvarchar](4000) NULL) ON [PRIMARY]

In [ ]:
declare @URL nvarchar(500) ='https://sessionize.com/api/v2/0scvywi2/view/sessions'
declare @response nvarchar(max)

exec sp_invoke_external_rest_endpoint @url=@URL, @response=@response OUTPUT
truncate table ConferenceSessions
insert into ConferenceSessions
select id, isnull(max(title),'') Title, isnull(max(description),'') description,  max(Name) MainSpeaker, STRING_AGG(Name, ', ') Speaker
from(SELECT JSON_VALUE(value, '$.id') id, JSON_VALUE(value, '$.title') title, JSON_VALUE(value, '$.description') description, JSON_VALUE(value, '$.startsAt') startsat, JSON_VALUE(value, '$.endsAt') endsat, speakers.Name
     FROM OPENJSON(JSON_QUERY(@response, '$.result[0].sessions'), 'strict $') as export
          OUTER APPLY
         OPENJSON(JSON_QUERY(value, '$.speakers'))
         WITH(name NVARCHAR(50) '$.name')speakers)a
group by id

In [ ]:
SELECT TOP 1 * FROM ConferenceSessions

In [ ]:
ALTER TABLE ConferenceSessions
ADD embeddings VECTOR(768)

In [ ]:
CREATE EXTERNAL MODEL ollama
WITH (
    LOCATION = 'https://ai-gpu.lab.bwdemo.io:443/api/embed',
    API_FORMAT = 'ollama',
    MODEL_TYPE = EMBEDDINGS,
    MODEL = 'nomic-embed-text'
);

In [ ]:
update ConferenceSessions set embeddings = AI_GENERATE_EMBEDDINGS(title + ' - ' + description,ollama)

In [ ]:
ALTER TABLE ConferenceSessions
ADD embeddings_16 vector(768, float16)

In [ ]:
ALTER DATABASE SCOPED CONFIGURATION SET PREVIEW_FEATURES = ON

In [ ]:
ALTER TABLE ConferenceSessions
ADD embeddings_16 vector(768, float16)

In [ ]:
update ConferenceSessions set embeddings_16 = AI_GENERATE_EMBEDDINGS(title + ' - ' + description,ollama)

In [ ]:
select top 1 (embeddings) as fp32_bytes, 
	(embeddings_16) as fp16_bytes
from 
	[dbo].[ConferenceSessions]